In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('history_traduzido.csv')

features_categoricas = [
    'genero', 'possui_companheiro', 'possui_dependentes', 'servico_telefonico', 
    'multiplas_linhas_telefonicas', 'servico_internet', 'seguranca_online', 
    'backup_online', 'protecao_dispositivos', 'suporte_tecnico', 'streaming_tv', 
    'streaming_filmes', 'tipo_contrato', 'faturamento_digital', 'metodo_pagamento'
]
features_numericas = ['idoso', 'tempo_permanencia', 'cobranca_mensal', 'cobranca_total_acumulada', 'renda_mensal']

encoder = LabelEncoder()
for col in features_categoricas:
    df[col] = encoder.fit_transform(df[col].astype(str))

df['cancelamento_servico'] = df['cancelamento_servico'].map({'Yes': 1, 'No': 0})

X_tabular = df[features_categoricas + features_numericas].values

modelo_nlp = SentenceTransformer('all-MiniLM-L6-v2')

feedbacks = df['feedback_cliente'].fillna("").tolist()
X_texto = modelo_nlp.encode(feedbacks, show_progress_bar=True)

X_final = np.hstack((X_tabular, X_texto))
y = df['cancelamento_servico'].values

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42, stratify=y)

modelo_xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1
)

modelo_xgb.fit(X_train, y_train)

previsoes = modelo_xgb.predict(X_test)
print(classification_report(y_test, previsoes, target_names=['Retido (0)', 'Cancelou (1)']))

ModuleNotFoundError: No module named 'sentence_transformers'

Support (1127): O modelo separou 1.127 clientes para a prova final (o conjunto de teste). Desses, 826 não cancelaram (Retido) e 301 cancelaram o serviço (Cancelou).

Precision (0.99): De todos os clientes que o seu algoritmo apontou o dedo e disse "vai cancelar", ele acertou 99% das vezes. A precisão avalia a qualidade do alerta.

Recall (0.96): Dos 301 clientes que de fato foram embora, o algoritmo conseguiu rastrear e identificar 96% deles. Os 4% restantes escaparam do radar (falsos negativos).

F1-Score (0.97): É a média harmônica entre a precisão e o recall, indicando um modelo quase perfeito. No mundo real da análise de dados, um F1-Score de 0.97 em churn é o primeiro sintoma de que o modelo tem informações privilegiadas.

In [ ]:
import pandas as pd

df_textos = pd.read_csv('history_traduzido.csv')

# clientes que cancelaram (o CSV mantém o 'Yes')
clientes_que_cancelaram = df_textos[df_textos['cancelamento_servico'] == 'Yes']

# 10 feedbacks completos para análise qualitativa
for index, texto in enumerate(clientes_que_cancelaram['feedback_cliente'].head(10)):
    print(f"Texto {index + 1}: {texto}\n")

Texto 1: I've been a customer with this company for over 2 and a half years now, and unfortunately I have decided to churn. The DSL internet service was reliable for the most part, but the monthly charges of $73.9 were a bit steep for the quality of service I was receiving. Additionally, having to mail in a check for payment was inconvenient and outdated. I ultimately decided to switch to a different provider that offered better value and more modern payment options.

Texto 2: I have been a customer with this company for 70 months and initially signed up for a two-year contract. I have been using their fiber optic internet service and have been satisfied with the speed and reliability. However, I recently decided to churn because I found a better deal with another provider. The monthly charges of $98.3 were reasonable for the service I received. I appreciated the convenience of automatic bank transfers for payment. Overall, my experience with this company was positive, but I ultimately

Vazamento de Dados Data Leakage
Por conta do cancel, churn etc, ele simplesmente leu a palavra "cancel" no texto e cravou que o cliente cancelou.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import plotly.graph_objects as go

df = pd.read_csv('history_traduzido.csv')

features_categoricas = [
    'genero', 'possui_companheiro', 'possui_dependentes', 'servico_telefonico', 
    'multiplas_linhas_telefonicas', 'servico_internet', 'seguranca_online', 
    'backup_online', 'protecao_dispositivos', 'suporte_tecnico', 'streaming_tv', 
    'streaming_filmes', 'tipo_contrato', 'faturamento_digital', 'metodo_pagamento'
]
features_numericas = [
    'idoso', 'tempo_permanencia', 'cobranca_mensal', 
    'cobranca_total_acumulada', 'renda_mensal'
]

encoder = LabelEncoder()
for col in features_categoricas:
    df[col] = encoder.fit_transform(df[col].astype(str))

# Converter a coluna alvo para 1 (Cancelou) e 0 (Retido)
df['cancelamento_servico'] = df['cancelamento_servico'].map({'Yes': 1, 'No': 0})

X = df[features_categoricas + features_numericas]
y = df['cancelamento_servico']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modelo_xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1
)

modelo_xgb.fit(X_train, y_train)

previsoes = modelo_xgb.predict(X_test)
print("--- RELATÓRIO DE PREVISÃO DE CHURN (SEM TEXTO) ---")
print(classification_report(y_test, previsoes, target_names=['Retido (0)', 'Cancelou (1)']))

#  Visualizar as variáveis de maior risco
importancias = modelo_xgb.feature_importances_
nomes_features = features_categoricas + features_numericas

df_importancia = pd.DataFrame({'Feature': nomes_features, 'Importancia': importancias})
df_importancia = df_importancia.sort_values(by='Importancia', ascending=True)

top_10 = df_importancia.tail(10)

fig = go.Figure(go.Bar(
    x=top_10['Importancia'],
    y=top_10['Feature'],
    orientation='h',
    marker=dict(color='royalblue')
))

fig.update_layout(
    title='Top 10 Variáveis que mais indicam risco de Churn',
    xaxis_title='Importância no Modelo',
    yaxis_title='Atributos do Cliente',
    height=500,
    template='plotly_white'
)

fig.show()

--- RELATÓRIO DE PREVISÃO DE CHURN (SEM TEXTO) ---
              precision    recall  f1-score   support

  Retido (0)       0.83      0.89      0.86       826
Cancelou (1)       0.62      0.50      0.55       301

    accuracy                           0.78      1127
   macro avg       0.72      0.69      0.71      1127
weighted avg       0.77      0.78      0.78      1127



In [ ]:
!pip install --upgrade nbformat


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np

df_silver = pd.read_csv('history_traduzido.csv')

def classificar_feedback(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return "Sem Feedback"
    t = str(texto).lower()
    
   
    if any(k in t for k in ["slow", "unreliable", "disappointed with the experience", "did not meet my expectations"]):
        return "Qualidade / Internet Lenta"
    elif any(k in t for k in ["check", "mail in", "payment method", "inconvenient", "outdated"]):
        return "Atrito no Método de Pagamento"
    elif any(k in t for k in ["steep", "too high", "better deal", "more affordable", "cost savings", "pricing", "better offer"]):
        return "Preço Alto / Oferta Concorrente"
    else:
        return "Satisfeito / Motivos Pessoais"

df_silver['motivo_feedback'] = df_silver['feedback_cliente'].apply(classificar_feedback)

mapa_sim_nao = {'Yes': 1, 'No': 0, 'Sim': 1, 'Não': 0, 1: 1, 0: 0, '1': 1, '0': 0}
df_silver['flag_companheiro'] = df_silver['possui_companheiro'].map(mapa_sim_nao).fillna(0).astype(int)
df_silver['flag_dependentes'] = df_silver['possui_dependentes'].map(mapa_sim_nao).fillna(0).astype(int)
df_silver['flag_idoso'] = df_silver['idoso'].map(mapa_sim_nao).fillna(0).astype(int)
df_silver['flag_churn'] = df_silver['cancelamento_servico'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

def definir_perfil_pessoal(row):
    idoso_str = "Idoso" if row['flag_idoso'] == 1 else "Não-Idoso"
    if row['flag_companheiro'] == 1 and row['flag_dependentes'] == 1:
        familia_str = "Com Cônjuge e Dependentes"
    elif row['flag_companheiro'] == 1 and row['flag_dependentes'] == 0:
        familia_str = "Apenas Cônjuge"
    elif row['flag_companheiro'] == 0 and row['flag_dependentes'] == 1:
        familia_str = "Apenas Dependentes (Filhos)"
    else:
        familia_str = "Solteiro / Sem Dependentes"
    return f"{idoso_str} - {familia_str}"

df_silver['perfil_pessoal'] = df_silver.apply(definir_perfil_pessoal, axis=1)

df_silver['comprometimento_renda_pct'] = np.where(
    df_silver['renda_mensal'] > 0,
    (df_silver['cobranca_mensal'] / df_silver['renda_mensal']) * 100,
    0
).round(2)

colunas_gold = [

    'flag_idoso', 'flag_companheiro', 'flag_dependentes', 'perfil_pessoal',

    'tipo_contrato', 'servico_internet', 'tempo_permanencia',
    'cobranca_mensal', 'cobranca_total_acumulada', 'renda_mensal', 'comprometimento_renda_pct',

    'motivo_feedback', 'flag_churn'
]

df_gold = df_silver[colunas_gold].copy()
df_gold['status_cliente'] = df_gold['flag_churn'].map({0: 'Bom Cliente (Retido)', 1: 'Cancelou (Churn)'})
print(f"Dimensões reduzidas com sucesso! Colunas na Gold: {df_gold.shape[1]}")
df_gold.head()

Dimensões reduzidas com sucesso! Colunas na Gold: 14


,flag_idoso,flag_companheiro,flag_dependentes,perfil_pessoal,tipo_contrato,servico_internet,tempo_permanencia,cobranca_mensal,cobranca_total_acumulada,renda_mensal,comprometimento_renda_pct,motivo_feedback,flag_churn,status_cliente
0,0,1,1,Não-Idoso - Com Cônjuge e Dependentes,One year,DSL,52.0,68.75,3482.85,6532.0,1.05,Atrito no Método de Pagamento,0,Bom Cliente (Retido)
1,0,1,1,Não-Idoso - Com Cônjuge e Dependentes,One year,DSL,33.0,73.90,2405.05,7634.0,0.97,Atrito no Método de Pagamento,1,Cancelou (Churn)
2,1,0,0,Idoso - Solteiro / Sem Dependentes,One year,No,22.0,20.05,417.00,3628.0,0.55,Atrito no Método de Pagamento,0,Bom Cliente (Retido)
3,1,1,1,Idoso - Com Cônjuge e Dependentes,Month-to-month,DSL,28.0,35.90,973.65,7851.0,0.46,Atrito no Método de Pagamento,0,Bom Cliente (Retido)
4,0,0,0,Não-Idoso - Solteiro / Sem Dependentes,Month-to-month,Fiber optic,2.0,70.30,144.00,1691.0,4.16,Atrito no Método de Pagamento,0,Bom Cliente (Retido)


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


resumo_pessoal = df_gold.groupby('perfil_pessoal').agg(
    total_clientes=('flag_churn', 'count'),
    total_cancelamentos=('flag_churn', 'sum'),
    taxa_churn=('flag_churn', lambda x: round(x.mean() * 100, 1)),
    receita_mensal_perdida=('cobranca_mensal', lambda x: round(x[df_gold.loc[x.index, 'flag_churn'] == 1].sum(), 2)),
    ticket_medio_churn=('cobranca_mensal', lambda x: round(x[df_gold.loc[x.index, 'flag_churn'] == 1].mean(), 2))
).reset_index()

resumo_pessoal = resumo_pessoal.sort_values(by='taxa_churn', ascending=True)

fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    subplot_titles=("Taxa Histórica de Cancelamento (%)", "Receita Mensal Perdida por Grupo ($)"),
    horizontal_spacing=0.08
)

fig.add_trace(go.Bar(
    x=resumo_pessoal['taxa_churn'],
    y=resumo_pessoal['perfil_pessoal'],
    orientation='h',
    text=resumo_pessoal['taxa_churn'].astype(str) + '%',
    textposition='auto',
    marker=dict(color='#EF553B'),
    name='Taxa de Churn (%)'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=resumo_pessoal['receita_mensal_perdida'],
    y=resumo_pessoal['perfil_pessoal'],
    orientation='h',
    text='$ ' + resumo_pessoal['receita_mensal_perdida'].astype(str),
    textposition='auto',
    marker=dict(color='#2a3f5f'),
    name='Receita Mensal Perdida ($)'
), row=1, col=2)

fig.update_layout(
    title='Diagnóstico Histórico: Perfil Pessoal e Familiar e Risco e Impacto na Receita',
    height=500,
    template='plotly_white',
    showlegend=False
)

fig.show()

In [ ]:
df_cancelados = df_gold[df_gold['flag_churn'] == 1]

motivos_para_remover = [
    'Satisfeito / Motivos Pessoais', 
    'Preço Alto / Oferta Concorrente'
]
df_cancelados = df_cancelados[~df_cancelados['motivo_feedback'].isin(motivos_para_remover)]

cruzamento_feedback = pd.crosstab(
    df_cancelados['perfil_pessoal'],
    df_cancelados['motivo_feedback'],
    normalize='index'
) * 100

cruzamento_feedback = cruzamento_feedback.round(1)

fig_fb = go.Figure()

cores_motivos = {
    'Qualidade / Internet Lenta': '#7570b3',
    'Atrito no Método de Pagamento': '#e7298a',
    'Sem Feedback': '#cccccc'
}

for motivo in cruzamento_feedback.columns:
    fig_fb.add_trace(go.Bar(
        name=motivo,
        y=cruzamento_feedback.index,
        x=cruzamento_feedback[motivo],
        orientation='h',
        text=cruzamento_feedback[motivo].astype(str) + '%',
        textposition='inside',
        marker_color=cores_motivos.get(motivo, 'royalblue')
    ))

fig_fb.update_layout(
    barmode='stack',
    title='Correlação Histórica: Perfil Pessoal vs. Motivo Real Citado no Feedback (Clientes que Cancelaram)',
    xaxis_title='Proporção dentro do Perfil (%)',
    yaxis_title='Perfil Pessoal do Cliente',
    height=520,
    template='plotly_white',
    legend_title='Causa do Cancelamento (Feedback)'
)

fig_fb.show()

In [ ]:

cols_corr = [
    'flag_churn', 'flag_idoso', 'flag_companheiro', 'flag_dependentes',
    'tempo_permanencia', 'cobranca_mensal', 'cobranca_total_acumulada',
    'renda_mensal', 'comprometimento_renda_pct'
]

nomes_amigaveis = [
    'Cancelou (Churn)', 'É Idoso', 'Possui Companheiro', 'Possui Dependentes',
    'Tempo de Permanência', 'Cobrança Mensal', 'Cobrança Acumulada',
    'Renda Mensal', 'Comprometimento da Renda (%)'
]

matriz_corr = df_gold[cols_corr].corr().round(2)

fig_corr = go.Figure(data=go.Heatmap(
    z=matriz_corr.values,
    x=nomes_amigaveis,
    y=nomes_amigaveis,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=matriz_corr.values,
    texttemplate="%{text}",
    textfont={"size": 11}
))

fig_corr.update_layout(
    title='Matriz de Correlação: Aspectos Pessoais, Financeiros e Cancelamento',
    height=600,
    template='plotly_white'
)

fig_corr.show()

In [ ]:
import plotly.graph_objects as go

corr_churn = matriz_corr['flag_churn'].drop('flag_churn').sort_values()


mapa_nomes = dict(zip(cols_corr, nomes_amigaveis))
nomes_eixo_y = [mapa_nomes[col] for col in corr_churn.index]

cores = ['#EF553B' if val > 0 else '#2a3f5f' for val in corr_churn.values]

fig_simples = go.Figure(go.Bar(
    x=corr_churn.values,
    y=nomes_eixo_y,
    orientation='h',
    marker_color=cores,
    text=corr_churn.values,
    textposition='outside'
))

fig_simples.update_layout(
    title='O que Puxou o Cancelamento para Cima (Vermelho) vs. O que Reteve o Cliente (Azul)',
    xaxis_title='Força da Relação com o Cancelamento (Negativo = Retém | Positivo = Cancela)',
    yaxis_title='Características do Cliente',
    xaxis=dict(range=[-0.45, 0.35], zeroline=True, zerolinewidth=2, zerolinecolor='black'),
    height=450,
    template='plotly_white'
)

fig_simples.show()

**1. Ações Imediatas por Perfil de Cliente**

* **Sénior Digital (Maior taxa de cancelamento):** O obstáculo principal é a dificuldade no pagamento. **Ação:** Criar um canal de ajuda prioritário para configurar o débito direto/cartão de crédito e enviar faturas mais simples e fáceis de ler.
* **Profissional Remoto (Maior perda de dinheiro):** Cancelam devido a internet lenta. **Ação:** Oferecer *upgrades* proativos de velocidade de internet e dar prioridade máxima no suporte técnico.
* **Famílias (Clientes mais fiéis):** **Ação:** Criar "Combos Familiares" e programas de fidelização para vender mais serviços (linhas extra, streaming) a quem já confia no serviço.

**2. Correções na Raiz do Problema**

* **Métodos de Pagamento:** Modernizar e remover imediatamente qualquer atrito no momento de pagar (acabar com processos manuais ou desatualizados).
* **Mensalidade vs. Rendimento:** Dar poder à equipa de retenção para oferecer planos mais baratos ou descontos temporários antes de perder o cliente para a concorrência.
* **Os Primeiros Meses:** O início é o momento de maior risco. Criar um programa de acompanhamento (*onboarding*) nos primeiros 3 a 6 meses para garantir que tudo funciona perfeitamente.

**3. Mudar a forma de ouvir o cliente**

* Não esperar que o cliente cancele para saber o que está mal. Fazer inquéritos rápidos de satisfação (NPS) ao longo do ano.
* No momento do cancelamento, obrigar a escolha de um motivo num menu de opções (Preço, Qualidade, Pagamento), para não dependermos apenas da leitura de textos soltos.